In [1]:
import torch
import numpy as np
from models.yolo3 import YOLOv3
from utils.bbox_iou import iou_wh
from utils.dataloaders import DatasetPascalVocYOLOv3, create_loader

In [2]:
SEED = 42
np.random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Ищем якоря

In [3]:
from utils.dataloaders import PascalVOC
from pathlib import Path
from torch.utils.data import random_split

In [4]:
ROOT = Path().cwd()
data = PascalVOC([ROOT.parent.joinpath("data/VOCdevkit/VOC2012"),
                  ROOT.parent.joinpath("data/VOCdevkit/VOC2007")])


In [5]:
data_ = data.create_data()
data_, _ = random_split(data_, [0.8, 0.2])

В наборе данных 22136 размеченных изображений.


In [6]:
def get_wh(data):
    wh = []
    for i in range(len(data)):
        size_img = data[i]["size_img"]
        for obj_ann in data[i]["objs_ann"]:
            x_min, y_min, x_max, y_max = obj_ann["bbox"]
            w = (x_max - x_min) / size_img[0]
            h = (y_max - y_min) / size_img[1]
            wh.append([w, h])

    return torch.tensor(wh)


In [7]:
wh  = get_wh(data_)

In [8]:
def kmeans_iou(boxes, k, n_iter=100, seed=42):
    rng = np.random.default_rng(seed)
    clusters = boxes[rng.choice(len(boxes), k, replace=False)]
    prev_assignments = None

    for _ in range(n_iter):
        ious = iou_wh(boxes, clusters) # Nxk,
        distances = 1 - ious
        assignments = distances.argmin(axis=1) # Nx1 или N,

        if prev_assignments is not None and (assignments == prev_assignments).all():
            break
        prev_assignments = assignments # Nx1

        for c in range(k): # 0 1 2 3 ...
            mask = assignments == c # Nx1 или N,
            if mask.any():
                clusters[c] = boxes[mask].mean(axis=0) # берем средние по тем значениям которые относятся к центроиду

    return clusters, assignments

In [9]:
acnhors, assignments = kmeans_iou(wh, k=9)

In [10]:
acnhors
areas = acnhors[:, 0] * acnhors[:, 1]
arg = areas.argsort()
acnhors = acnhors[arg]

In [11]:
acnhors

tensor([[0.0560, 0.1014],
        [0.1692, 0.1596],
        [0.1064, 0.3165],
        [0.3327, 0.3056],
        [0.1970, 0.5305],
        [0.3435, 0.6849],
        [0.6961, 0.4395],
        [0.5494, 0.8021],
        [0.8777, 0.8453]])

# Проверка датасета

In [12]:
kwargs = {"list_size" :[52, 26, 13], "achors" : acnhors}

data = create_loader(path=
                        [
                            ROOT.parent.joinpath("data/VOCdevkit/VOC2012"),
                            ROOT.parent.joinpath("data/VOCdevkit/VOC2007")
                        ],
                        split=[0.8, 0.2],
                        batch_size=1,
                        workers=0,
                        pin_memory=True,
                        shuffle_train=True,
                        image_size=416,
                        dataset_class=DatasetPascalVocYOLOv3,
                        **kwargs,
                    )


В наборе данных 22136 размеченных изображений.


In [13]:
train_loader, val_loader = data

In [14]:
b = next(iter(train_loader))

In [15]:
b[0].shape

torch.Size([1, 3, 416, 416])

In [16]:
b[1].shape

torch.Size([1, 52, 52, 3, 25])

In [17]:
a = torch.tensor([[1, 2], [0, 1]])

mask = torch.any((a > 0), dim=-1)

In [18]:
mask

tensor([True, True])

# Проверка лосса

In [19]:
from loss.loss import YOLOv3Loss
from models.yolo3 import YOLOv3

In [20]:
model = YOLOv3(num_classes=20)
loss_model = YOLOv3Loss(anchors=acnhors)

In [21]:
pred = model(b[0])

In [22]:
loss, loss_value = loss_model(pred, b[1:])

In [23]:
loss

tensor(825.5768, grad_fn=<DivBackward0>)

# Обучение

In [2]:
import torch
import numpy as np

from models.yolo3 import YOLOv3
from loss.loss import YOLOv3Loss
from torch.optim import SGD
from torch.optim.lr_scheduler import LambdaLR

from utils.dataloaders import create_loader, PascalVOC, DatasetPascalVocYOLOv3
from utils.nms import nms_yolov3
from pathlib import Path
from tqdm.notebook import tqdm
from utils.bbox_iou import box_iou,  iou_wh
from metrics.metrics import ap_per_class
from torch.utils.data import random_split

In [3]:
SEED = 42
BATCH_SIZE = 1
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
ROOT = Path().cwd()
data = PascalVOC(dirs=[ROOT.parent.joinpath("data/VOCdevkit/VOC2012"),
                       ROOT.parent.joinpath("data/VOCdevkit/VOC2007")])

In [5]:
data = data.create_data()
data_train, _ = random_split(data, [0.8, 0.2])

В наборе данных 22136 размеченных изображений.


In [6]:
def get_wh(data):
    wh = []
    for i in range(len(data)):
        size_img = data[i]["size_img"]
        for obj_ann in data[i]["objs_ann"]:
            x_min, y_min, x_max, y_max = obj_ann["bbox"]
            w = (x_max - x_min)/size_img[0]
            h = (y_max - y_min)/size_img[1]
            wh.append([w, h])

    return torch.tensor(wh)
wh = get_wh(data_train)

In [7]:
def kmeans_iou(boxes, k, n_iter=100, seed=42):
    rng = np.random.default_rng(seed)
    clusters = boxes[rng.choice(len(boxes), k, replace=False)]
    prev_assignments = None

    for _ in range(n_iter):
        ious = iou_wh(boxes, clusters)
        distances = 1 - ious
        assignments = distances.argmin(axis=1)

        if prev_assignments is not None and (assignments == prev_assignments).all():
            break

        prev_assignments = assignments

        for c in range(k):
            mask = assignments == c
            if mask.any():
                clusters[c] = boxes[mask].mean(axis=0)

    return clusters, assignments

anchors, assignments = kmeans_iou(wh, k=9)
areas = anchors[:, 0] * anchors[:, 1]
arg = areas.argsort()
anchors = anchors[arg]
anchors

tensor([[0.0560, 0.1014],
        [0.1692, 0.1596],
        [0.1064, 0.3165],
        [0.3327, 0.3056],
        [0.1970, 0.5305],
        [0.3435, 0.6849],
        [0.6961, 0.4395],
        [0.5494, 0.8021],
        [0.8777, 0.8453]])

In [8]:
kwargs = {"list_size" :[52, 26, 13], "achors" : anchors}

train_loader, val_loader = create_loader(
    [
        ROOT.parent.joinpath("data/VOCdevkit/VOC2012"),
        ROOT.parent.joinpath("data/VOCdevkit/VOC2007")
    ],
    split=[0.8, 0.2],
    batch_size=BATCH_SIZE,
    workers=0,
    image_size=416,
    pin_memory=True,
    shuffle_train=True,
    dataset_class=DatasetPascalVocYOLOv3,
    **kwargs,

)


В наборе данных 22136 размеченных изображений.


In [9]:
model = YOLOv3(num_classes=20)
loss_model = YOLOv3Loss(anchors=anchors)
optim = SGD(params=model.parameters(), lr=0.001, weight_decay=5e-4, momentum=0.9)

WARMUP_EPOCHS = 20
PLATEAU_EPOCHS = 10
EPOCHS = 300

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return epoch / WARMUP_EPOCHS
    if epoch < WARMUP_EPOCHS + PLATEAU_EPOCHS:
        return 1.0

    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))

sheduler = LambdaLR(optim, lr_lambda)

In [10]:
def compute_tp(pred_boxes, pred_scores, pred_classes, gt_boxes, gt_classes, iou_threshold=0.5):
    order = pred_scores.argsort(descending=True)
    pred_boxes = pred_boxes[order]
    pred_classes = pred_classes[order]
    pred_scores = pred_scores[order]

    tp = torch.zeros(len(pred_boxes))
    matched_gt = set()

    for i in range(len(pred_boxes)):
        same_class_mask = gt_classes == pred_classes[i]
        if not same_class_mask.any():
            continue

        same_class_idx = same_class_mask.nonzero().squeeze(1)
        ious = box_iou(pred_boxes[i:i+1], gt_boxes[same_class_mask])
        best_iou, best_local_idx = ious.max(dim=1)
        best_gt_idx = same_class_idx[best_local_idx].item()

        if best_iou.item() >= iou_threshold and best_gt_idx not in matched_gt:
            tp[i] = 1
            matched_gt.add(best_gt_idx)

    return tp, pred_scores, pred_classes

        

In [11]:
train_loss_history, val_loss_history, mAP_50_history = list(), list(), list()
best_train_loss, best_mAP = float("inf"), 0

loop = tqdm(range(EPOCHS), leave=True, desc="Training YOLOv3")
for epoch in loop:
    loss_epoch = []
    model.train()
    loop_train = tqdm(enumerate(train_loader), leave=False, desc=f"Train [{epoch+1}/{EPOCHS}]", total=len(train_loader))

    for i, (imgs, target) in loop_train:
        pred = model(imgs)
        loss, loss_value = loss_model(pred, target)
        loss_epoch.append(loss_value)

        optim.zero_grad()
        loss.backward()
        optim.step()

        if i % 10 == 0:
            loop_train.set_postfix(loss=loss_epoch[-1])

    train_loss_history.append(np.mean(loss_epoch))
    best_train_loss = min(train_loss_history, best_train_loss)
    loop.set_postfix(best_train_loss=best_train_loss, best_mAP=best_mAP)

    loss_epoch = []
    all_tp, all_conf, all_pred_cls, all_target_cls = [], [], [], []
    model.eval()
    loop_val = tqdm(enumerate(val_loader), leave=False, desc=f"Validation [{epoch+1}/{EPOCHS}]", total=len(val_loader))
    with torch.no_grad():
        for i, (imgs, target) in loop_val:
            pred = model(imgs)
            flatten_pred = loss_model.forward_eval(pred)
            filtered_pred = nms_yolov3(flatten_pred)

            for b in range(len(filtered_pred)):
                det = filtered_pred[b]
                gt = target[b]

                if det.shape[0] == 0:
                    all_target_cls.append(gt)
                    continue

                gt_boxes_xyxy = gt[:, :4]
                gt_classes = gt[:, 4]
                tp, conf, pred_cls = compute_tp(det[:, :4], det[:, 4], det[:, 5], gt_boxes_xyxy, gt_classes)

                all_tp.append(tp.cpu().numpy())
                all_conf.append(conf.cpu().numpy())
                all_pred_cls.append(pred_cls.cpu().numpy())
                all_target_cls.append(gt_classes.cpu().numpy())

    if len(all_tp) != 0:
        tp = np.concatenate(all_tp).reshape(-1, 1)
        conf = np.concatenate(all_conf)
        pred_cls = np.concatenate(all_pred_cls)
        target_cls = np.concatenate(all_target_cls)

        ap = ap_per_class(tp, conf, pred_cls, target_cls)
        mAP_50 = ap[:, 0].mean()
    else:
        mAP_50 = 0

    mAP_50_history.append(mAP_50)
    best_mAP = max(mAP_50, best_mAP)
    loop.set_postfix(best_train_loss=best_train_loss, best_mAP=best_mAP)
    sheduler.step()


Training YOLOv3:   0%|          | 0/300 [00:00<?, ?it/s]

Train [1/300]:   0%|          | 0/17709 [00:00<?, ?it/s]

KeyboardInterrupt: 